# Example of Toolbox method

## Set-up

In [ ]:
import os
# while not "src" in os.listdir(): # I use this to change my directory to access the src and data folders from in here
#     os.chdir("..") 

import networkx as nx
import numpy as np
import scipy as sp
import pandas as pd
import pprint
import copy

import mygene # allow us to convert from systematic to standard

import matplotlib.pyplot as plt

from reading_in import read_in_proteins
from community_finding import partition_graph, partition_graph_size, eliminate_small_communities, get_adjacent_communities, size_distribution_of_communities
from scoring_and_selection import text_selection, simple_text_selection, selecting_high_frequency_intermediary_nodes, compress_important_nodes, trim_nodes_dictionary
from STM.stm import run_stm_algorithm
from Exclusion.TEO_method import calculate_TEO


FILEPATH = "protein.links.v12.0.txt.gz"
G = read_in_proteins(include_essential_proteins=False, connected=True)

In [59]:
def eliminate_small_communities(G: nx.Graph, threshold: int, community_finding_algorithm, seed: int | None = 42, resolution: float = 1.0) -> list[set]:
    try:
        communities_list = community_finding_algorithm(G, seed=seed, resolution=resolution)
    except:
        communities_list = community_finding_algorithm(G, resolution=resolution)
    communities_list.sort(key=len)
    filtered_list = [x for x in communities_list if len(x) >= threshold]
    return filtered_list

In [57]:
# Sets up centrality measures.
degcent = nx.degree_centrality(G)
eigcent = nx.eigenvector_centrality(G)
ClosenessCent = nx.closeness_centrality(G) 
Betweencent = nx.betweenness_centrality(G)
cents = [degcent, eigcent, ClosenessCent, Betweencent]

In [60]:
# Sets up algorithm
algorithms = lambda G, resolution, seed : run_stm_algorithm(G, threshold=0.05)

## Execution

In [61]:
# Output of this all is gonna come out as a dictionary, which shows the key as a high centrality node, and the value as the 
# high relative frequency nodes that connect pep5 to the high centrality node, listed also with their relative frequency
communities = eliminate_small_communities(G, community_finding_algorithm = algorithms, threshold=3, seed=986578, resolution=0.8)
adjacent_communities = get_adjacent_communities(G, communities)
listthing = []
for meas in cents:
    listthing.append(selecting_high_frequency_intermediary_nodes(G, adjacent_communities, meas, exclude_representatives=True, relative_frequency=True))
draft_nodes_list_stm = compress_important_nodes(listthing)

Network Stats: 4209 Nodes, 41037 Edges
Executing Phase 1: Computing Signal Transduction Matrices...
Executing Phase 2: Generating Core Pre-clusters...
Executing Phase 3: Merging overlapping structures (Threshold >= 0.05)...

SUCCESS: Extracted 274 Total Functional Modules:
Module #1 (50 proteins): ['Q0010', 'Q0017', 'Q0032', 'Q0045', 'Q0080', 'Q0092', 'Q0142', 'Q0143', 'Q0182', 'Q0297', 'YAL015C', 'YBL013W', 'YBR028C', 'YBR255C-A', 'YCR095W-A', 'YDL002C', 'YDR174W', 'YDR263C', 'YDR296W', 'YDR349C', 'YDR350C', 'YEL009C', 'YEL059C-A', 'YER045C', 'YER153C', 'YFL036W', 'YGL107C', 'YGL133W', 'YGR150C', 'YHR120W', 'YIL036W', 'YIL070C', 'YJL147C', 'YJL209W', 'YJR144W', 'YKL011C', 'YLR139C', 'YMR035W', 'YMR072W', 'YMR150C', 'YMR166C', 'YMR228W', 'YMR257C', 'YNL215W', 'YNR045W', 'YOL023W', 'YOR354C', 'YPL005W', 'YPR052C', 'YPR153W']
Module #2 (1760 proteins): ['Q0050', 'Q0055', 'Q0060', 'Q0065', 'Q0070', 'Q0075', 'Q0085', 'Q0105', 'Q0110', 'Q0115', 'Q0120', 'Q0130', 'Q0140', 'Q0160', 'Q0250', '

In [62]:
pprint.pprint(draft_nodes_list_stm)

{'YAL054C': {('YBR221C', 0.15789473684210525),
             ('YDR207C', 0.15789473684210525),
             ('YLR304C', 0.13157894736842105),
             ('YOR142W', 0.15789473684210525)},
 'YBL019W': {('YDR419W', 0.391304347826087),
             ('YER095W', 0.17391304347826086),
             ('YGL191W', 0.391304347826087),
             ('YML095C', 0.08695652173913043)},
 'YBL091C': {('YDL136W', 0.2358490566037736),
             ('YDL191W', 0.2358490566037736),
             ('YGR034W', 0.19811320754716982),
             ('YLR344W', 0.2169811320754717)},
 'YBR018C': {('YDR477W', 0.14049586776859505),
             ('YEL009C', 0.12396694214876033),
             ('YEL021W', 0.10743801652892562),
             ('YPR119W', 0.12396694214876033)},
 'YBR026C': {('YER061C', 0.3333333333333333),
             ('YGL205W', 0.3333333333333333),
             ('YLR281C', 0.6666666666666666)},
 'YBR053C': {('YLR178C', 1.0)},
 'YBR249C': {('YBR221C', 0.08571428571428572),
             ('YCL030C', 0.185714

### And from here, one can apply exclusion methods to create a more curated list. 

In [ ]:
expression_filepath = "Exclusion/GSE3431_gene_expression.csv"
gaf_filepath = "Exclusion/ppi-data/go/YEAST-mod.gaf"
go_obo_filepath = "Exclusion/ppi-data/go/go-basic.obo"
teo = calculate_TEO(G, expression_filepath,gaf_filepath,go_obo_filepath)

Expression profiles for 3,948/4,209 graph nodes
GAF annotation rows retained for aspect P: 42,115
Graph nodes with at least one matched GO annotation: 4,202/4,209
Matched annotation rows: 26,763
Calculated Wang GO similarities for 40,992/41,037 graph edges


In [64]:
modified_nodes_list = copy.deepcopy(draft_nodes_list_stm)
for keys in list(draft_nodes_list_stm.keys()):
    for item in draft_nodes_list_stm[keys]:
        a = simple_text_selection(item[0])
        try:
            b = teo[item[0]]
        except:
            print(item[0], "error")
            b = 0
#        print(a)
        if not a or teo[item[0]] > 10:
            modified_nodes_list.pop(keys, None)
            break
pprint.pprint(modified_nodes_list)

{'YBR053C': {('YLR178C', 1.0)},
 'YBR258C': {('YAR003W', 0.18181818181818182),
             ('YBR175W', 0.18181818181818182),
             ('YDR469W', 0.18181818181818182),
             ('YLR015W', 0.18181818181818182)},
 'YCR079W': {('YDL006W', 0.29411764705882354),
             ('YER178W', 0.17647058823529413),
             ('YLR356W', 0.5882352941176471),
             ('YOR323C', 0.11764705882352941)},
 'YDL049C': {('YNL322C', 1.0)},
 'YDL159W-A': {('Q0255', 1.0)},
 'YDL216C': {('YFR010W', 0.28888888888888886),
             ('YHR200W', 0.28888888888888886),
             ('YJR099W', 0.24444444444444444),
             ('YLR421C', 0.24444444444444444)},
 'YDL239C': {('YDR260C', 0.125),
             ('YGL240W', 0.125),
             ('YGR225W', 0.125),
             ('YOR042W', 0.75)},
 'YDR249C': {('YBR137W', 1.0)},
 'YDR262W': {('YMR075W', 0.6666666666666666)},
 'YDR273W': {('YOR042W', 1.0)},
 'YDR479C': {('YGR004W', 0.2222222222222222),
             ('YLR324W', 0.6666666666666666),
   